# Experiment 5 — Unsupervised Learning

**Course:** ENCS5141  
**Student:** Nadeen — ID: 1220044  

This notebook covers:
- K-Means Clustering
- DBSCAN Clustering
- Gaussian Mixture Model (GMM) Clustering
- Quantitative Evaluation
- **ToDo 1:** Two Moons Dataset
- **ToDo 2:** K-Means vs K-Medoids on Blobs
- **ToDo 3:** Image Segmentation with K-Means

---
## 0. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn import metrics
from sklearn_extra.cluster import KMedoids   # pip install scikit-learn-extra

print('All imports successful.')

---
## 1. Data Generation — Blobs

In [ ]:
# Generate a Gaussian 2D dataset (blobs)
centers = [[1, 1], [-1, -1], [1, -1]]
X, labels_true = make_blobs(
    n_samples=750, centers=centers, cluster_std=0.4, random_state=0)
X = StandardScaler().fit_transform(X)

# Visualise blobs (unlabelled and labelled)
plt.figure(figsize=(12, 6))
colormap = np.array(['red', 'lime', 'black'])

plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1])
plt.title('Blobs (no labels)')

plt.subplot(1, 2, 2)
plt.scatter(X[:, 0], X[:, 1], c=colormap[labels_true], s=40)
plt.title('Blobs with Ground-Truth Labels')

plt.suptitle('Figure 1.1 — Generated Blobs', fontsize=13)
plt.tight_layout()
plt.show()

---
## 2. K-Means Clustering on Blobs

In [ ]:
# ── Fit K-Means with K = 3 ──────────────────────────────────────────────────
model = KMeans(n_clusters=3, random_state=0)
model.fit(X)
predY = np.choose(model.labels_, [0, 1, 2]).astype(np.int64)
k_means_cluster_centers = model.cluster_centers_

# ── Visualise ───────────────────────────────────────────────────────────────
plt.figure(figsize=(12, 6))
colormap = np.array(['red', 'lime', 'black'])

plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], c=colormap[labels_true], s=40)
plt.title('Blobs with GT Labels')

plt.subplot(1, 2, 2)
plt.scatter(X[:, 0], X[:, 1], c=colormap[predY], s=40)
plt.scatter(k_means_cluster_centers[:, 0], k_means_cluster_centers[:, 1],
            marker='*', c='orange', s=200, zorder=5, label='Centroids')
plt.title('Blobs after K-Means (K=3)')
plt.legend()

plt.suptitle('Figure 1.2 — K-Means Clustering Results (K=3)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Try different K values ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
colormap_ext = plt.cm.tab10

for ax, k in zip(axes, [2, 3, 4, 6]):
    km = KMeans(n_clusters=k, random_state=0)
    km.fit(X)
    ax.scatter(X[:, 0], X[:, 1], c=km.labels_, cmap=colormap_ext, s=20)
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
               marker='*', c='orange', s=200, zorder=5)
    sil = metrics.silhouette_score(X, km.labels_)
    ax.set_title(f'K={k}  |  Silhouette={sil:.3f}')

plt.suptitle('K-Means with Different K Values', fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. DBSCAN Clustering on Blobs

In [ ]:
# ── Fit DBSCAN ───────────────────────────────────────────────────────────────
db = DBSCAN(eps=0.3, min_samples=10).fit(X)   # tuned for blobs
labels_db = db.labels_

n_clusters_ = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise_    = list(labels_db).count(-1)
print(f'Estimated number of clusters: {n_clusters_}')
print(f'Estimated number of noise points: {n_noise_}')

# ── Visualise core vs border vs noise ────────────────────────────────────────
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], c=colormap[labels_true], s=40)
plt.title('Blobs with GT Labels')

plt.subplot(1, 2, 2)
unique_labels       = set(labels_db)
core_samples_mask   = np.zeros_like(labels_db, dtype=bool)
core_samples_mask[db.core_sample_indices_] = True
colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]

for k, col in zip(unique_labels, colors):
    col = [0, 0, 0, 1] if k == -1 else col
    mask = labels_db == k
    # core points — large
    xy = X[mask & core_samples_mask]
    plt.plot(xy[:, 0], xy[:, 1], 'o',
             markerfacecolor=tuple(col), markeredgecolor='k', markersize=14)
    # border / noise — small
    xy = X[mask & ~core_samples_mask]
    plt.plot(xy[:, 0], xy[:, 1], 'o',
             markerfacecolor=tuple(col), markeredgecolor='k', markersize=6)

plt.title(f'DBSCAN — estimated clusters: {n_clusters_}')
plt.suptitle('Figure 1.3 — DBSCAN Clustering Results', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Effect of varying eps & min_samples ──────────────────────────────────────
params = [(0.1, 5), (0.3, 10), (0.5, 10), (0.5, 20)]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, (eps, mpts) in zip(axes, params):
    db_ = DBSCAN(eps=eps, min_samples=mpts).fit(X)
    lbl = db_.labels_
    nc  = len(set(lbl)) - (1 if -1 in lbl else 0)
    nn  = list(lbl).count(-1)
    ax.scatter(X[:, 0], X[:, 1], c=lbl, cmap='Spectral', s=15)
    ax.set_title(f'eps={eps}, minPts={mpts}\nclusters={nc}, noise={nn}')

plt.suptitle('DBSCAN — Sensitivity to Hyperparameters', fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. Gaussian Mixture Model (GMM) Clustering on Blobs

In [ ]:
# ── Fit GMM with n_components = 3 ────────────────────────────────────────────
gmm = GaussianMixture(n_components=3, random_state=0)
gmm.fit(X)
y_cluster_gmm = gmm.predict(X)

# ── Visualise ────────────────────────────────────────────────────────────────
plt.figure(figsize=(12, 6))
colormap = np.array(['red', 'lime', 'black'])

plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], c=colormap[labels_true], s=40)
plt.title('Blobs with GT Labels')

plt.subplot(1, 2, 2)
plt.scatter(X[:, 0], X[:, 1], c=colormap[y_cluster_gmm], s=40)
plt.title('Blobs after GMM (n_components=3)')

plt.suptitle('Figure 1.4 — GMM Clustering Results', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Try different n_components ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, nc in zip(axes, [2, 3, 4, 6]):
    g = GaussianMixture(n_components=nc, random_state=0).fit(X)
    yg = g.predict(X)
    ax.scatter(X[:, 0], X[:, 1], c=yg, cmap='tab10', s=20)
    sil = metrics.silhouette_score(X, yg)
    ax.set_title(f'n_comp={nc}  |  Silhouette={sil:.3f}')

plt.suptitle('GMM with Different n_components', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Quantitative Evaluation — Blobs

In [ ]:
def print_metrics(name, y_true, y_pred, X):
    print(f'──────────────────────────────────')
    print(f'{name} Evaluation Measures\n')
    print(f'  Homogeneity:              {metrics.homogeneity_score(y_true, y_pred):.3f}')
    print(f'  Completeness:             {metrics.completeness_score(y_true, y_pred):.3f}')
    print(f'  V-measure:                {metrics.v_measure_score(y_true, y_pred):.3f}')
    print(f'  Adjusted Rand Index:      {metrics.adjusted_rand_score(y_true, y_pred):.3f}')
    print(f'  Adjusted Mutual Info:     {metrics.adjusted_mutual_info_score(y_true, y_pred):.3f}')
    try:
        sil = metrics.silhouette_score(X, y_pred)
        print(f'  Silhouette Coefficient:   {sil:.3f}')
    except Exception:
        print('  Silhouette Coefficient:   N/A (fewer than 2 clusters)')
    print()

print_metrics('DBSCAN',  labels_true, labels_db,     X)
print_metrics('K-Means', labels_true, predY,          X)
print_metrics('GMM',     labels_true, y_cluster_gmm,  X)

---
## 6. ToDo 1 — Two Moons Dataset

Repeat all three clustering methods on the non-Gaussian **two-moons** dataset and compare results qualitatively and quantitatively.

In [ ]:
# ── Generate two-moons data ───────────────────────────────────────────────────
X_moons, labels_moons = make_moons(n_samples=500, noise=0.1, random_state=42)

plt.figure(figsize=(5, 4))
plt.scatter(X_moons[:, 0], X_moons[:, 1],
            c=['red' if l == 0 else 'blue' for l in labels_moons], s=20)
plt.title('Two Moons — Ground Truth')
plt.tight_layout()
plt.show()

In [ ]:
# ── K-Means on Two Moons (K=2) ────────────────────────────────────────────────
km_moons = KMeans(n_clusters=2, random_state=0)
km_moons.fit(X_moons)
pred_km_moons = km_moons.labels_

# ── DBSCAN on Two Moons ───────────────────────────────────────────────────────
db_moons  = DBSCAN(eps=0.15, min_samples=5).fit(X_moons)
pred_db_moons = db_moons.labels_
nc_moons  = len(set(pred_db_moons)) - (1 if -1 in pred_db_moons else 0)
print(f'DBSCAN on two-moons: {nc_moons} clusters, {list(pred_db_moons).count(-1)} noise points')

# ── GMM on Two Moons ──────────────────────────────────────────────────────────
gmm_moons = GaussianMixture(n_components=2, random_state=0)
gmm_moons.fit(X_moons)
pred_gmm_moons = gmm_moons.predict(X_moons)

In [ ]:
# ── DBSCAN on Two Moons — fixed visualization ─────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(22, 4))
titles = ['GT', 'K-Means (K=2)', 'DBSCAN (eps=0.15, minPts=5)', 'GMM (n=2)']

# GT
axes[0].scatter(X_moons[:, 0], X_moons[:, 1],
                c=['red' if l==0 else 'blue' for l in labels_moons], s=15)
axes[0].set_title(titles[0])

# K-Means
axes[1].scatter(X_moons[:, 0], X_moons[:, 1],
                c=['red' if l==0 else 'blue' for l in pred_km_moons], s=15)
axes[1].set_title(titles[1])

# DBSCAN — handle noise points explicitly
db_colors = []
for l in pred_db_moons:
    if l == -1:
        db_colors.append('black')   # noise → black
    elif l == 0:
        db_colors.append('red')
    else:
        db_colors.append('blue')
axes[2].scatter(X_moons[:, 0], X_moons[:, 1], c=db_colors, s=15)
axes[2].set_title(titles[2])

# GMM
axes[3].scatter(X_moons[:, 0], X_moons[:, 1],
                c=['red' if l==0 else 'blue' for l in pred_gmm_moons], s=15)
axes[3].set_title(titles[3])

plt.suptitle('Two Moons — Qualitative Comparison', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Quantitative comparison ───────────────────────────────────────────────────
print('====== Two Moons Dataset ======')
print_metrics('K-Means', labels_moons, pred_km_moons,  X_moons)
print_metrics('DBSCAN',  labels_moons, pred_db_moons,  X_moons)
print_metrics('GMM',     labels_moons, pred_gmm_moons, X_moons)

### Discussion — Blobs vs Two Moons

| Algorithm | Blobs | Two Moons |
|-----------|-------|----------|
| **K-Means** | Excellent — blobs are roughly spherical | Poor — cannot capture crescent shapes; creates straight-line boundaries |
| **DBSCAN** | Very good — handles density variation | Excellent — naturally follows the curved density of each moon |
| **GMM** | Excellent — models elliptical Gaussians well | Moderate — assumes Gaussian components, struggles with non-convex shapes |

**Key insight:** DBSCAN is the clear winner on two-moons because it is a *density-based* method that does not assume any particular cluster shape, while K-Means and GMM assume (approximately) convex/Gaussian clusters.

---
## 7. ToDo 2 — K-Means vs K-Medoids on Blobs

K-Means minimises the total squared error to the cluster *mean* (which may not be a real data point).  
K-Medoids minimises the total dissimilarity to the cluster *medoid* — an actual data point. This makes K-Medoids more robust to outliers.

In [ ]:
# ── K-Medoids on blobs ────────────────────────────────────────────────────────
kmed = KMedoids(n_clusters=3, random_state=0)
kmed.fit(X)
pred_kmed = kmed.labels_
kmed_centers = kmed.cluster_centers_

# ── Visual comparison ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# GT
axes[0].scatter(X[:, 0], X[:, 1], c=colormap[labels_true], s=20)
axes[0].set_title('Ground Truth')

# K-Means
axes[1].scatter(X[:, 0], X[:, 1], c=colormap[predY], s=20)
axes[1].scatter(k_means_cluster_centers[:, 0], k_means_cluster_centers[:, 1],
                marker='*', c='orange', s=300, zorder=5, label='Centroids')
axes[1].set_title('K-Means (K=3)')
axes[1].legend()

# K-Medoids
axes[2].scatter(X[:, 0], X[:, 1], c=colormap[pred_kmed % 3], s=20)
axes[2].scatter(kmed_centers[:, 0], kmed_centers[:, 1],
                marker='D', c='orange', s=150, zorder=5, label='Medoids')
axes[2].set_title('K-Medoids (K=3)')
axes[2].legend()

plt.suptitle('ToDo 2 — K-Means vs K-Medoids on Blobs', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Quantitative comparison ───────────────────────────────────────────────────
print('====== Blobs — K-Means vs K-Medoids ======')
print_metrics('K-Means',   labels_true, predY,      X)
print_metrics('K-Medoids', labels_true, pred_kmed,  X)

### Discussion — K-Means vs K-Medoids

- On clean blobs data (low noise, few outliers) both algorithms achieve very similar scores because the mean and medoid tend to fall close to each other.
- **K-Medoids** is inherently more robust to outliers: because the cluster centre must be an actual data point, a single extreme outlier cannot drag the centre far from the true cluster mass the way it can with K-Means.
- **K-Means** is generally faster (O(n·k·i) vs O(n²·k·i) for classic K-Medoids), so it is preferred when the dataset is large and free of severe outliers.

---
## 8. ToDo 3 — Image Segmentation with K-Means

Segment a colour image into K = 2, 5, and 10 clusters using K-Means.  
Each pixel's (R, G, B) values are treated as a 3-D data point.

**Replace** `'your_image.jpg'` below with the path to your own image.

In [ ]:
# ── Helper: segment image with K-Means ───────────────────────────────────────
def segment_image(img_path: str, k_values=(2, 5, 10)):
    """
    Load a colour image, apply K-Means segmentation for each K in k_values,
    and display the original alongside all segmented versions.
    """
    img = mpimg.imread(img_path)
    
    # Normalise to [0, 1] if the image is uint8
    if img.dtype == np.uint8:
        img = img.astype(np.float32) / 255.0
    
    # Drop alpha channel if present
    if img.ndim == 3 and img.shape[2] == 4:
        img = img[:, :, :3]
    
    h, w, c = img.shape
    pixels = img.reshape(-1, c)   # (h*w, 3)

    fig, axes = plt.subplots(1, len(k_values) + 1, figsize=(5 * (len(k_values) + 1), 5))

    axes[0].imshow(img)
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    for ax, k in zip(axes[1:], k_values):
        km = KMeans(n_clusters=k, random_state=0, n_init='auto')
        km.fit(pixels)
        # Replace each pixel by its cluster-centre colour
        segmented = km.cluster_centers_[km.labels_].reshape(h, w, c)
        segmented = np.clip(segmented, 0, 1)
        ax.imshow(segmented)
        ax.set_title(f'K = {k}')
        ax.axis('off')

    plt.suptitle('Image Segmentation using K-Means', fontsize=14)
    plt.tight_layout()
    plt.show()
    print(f'Image shape: {h}×{w}, {c} channels')


# ── Run segmentation — replace path with your own image ──────────────────────
segment_image('im.jpg', k_values=(2, 5, 10))

In [ ]:
# ── Demo with a synthetic test image (runs without any external file) ─────────
# Remove / comment this cell and call segment_image() above with your own photo.

from matplotlib.colors import hsv_to_rgb

# Create a simple synthetic colour image (200×200 with coloured patches)
demo = np.zeros((200, 200, 3), dtype=np.float32)
demo[:100, :100]  = [1.0, 0.2, 0.2]   # red
demo[:100, 100:]  = [0.2, 0.8, 0.2]   # green
demo[100:, :100]  = [0.2, 0.2, 1.0]   # blue
demo[100:, 100:]  = [1.0, 1.0, 0.0]   # yellow
# Add some noise
rng = np.random.default_rng(42)
demo = np.clip(demo + rng.normal(0, 0.08, demo.shape).astype(np.float32), 0, 1)

# Save temp file then segment
import tempfile, os
tmp = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
tmp_path = tmp.name
tmp.close()  # Close handle so Windows allows later deletion
plt.imsave(tmp_path, demo)
segment_image(tmp_path, k_values=(2, 5, 10))
try:
    os.unlink(tmp_path)
except PermissionError:
    pass

### Discussion — Image Segmentation

| K | Observation |
|---|-------------|
| **K = 2** | Only two colours remain — the image is reduced to a binary-like silhouette. Fine details and gradients are completely lost. Useful for simple foreground/background separation. |
| **K = 5** | More colour regions are preserved; major objects become distinguishable. A good balance between compression and perceptual quality for many real-world images. |
| **K = 10** | The image looks noticeably closer to the original; subtle colour transitions and textures are better captured, though still simplified compared to the full-colour original. |

**General observation:** Increasing K reduces compression but increases detail retention. K-Means image segmentation is essentially *lossy colour quantisation* — it maps the full continuous colour space onto K representative colours.